# AR order scale-selection pipeline with an $L^1$ weight and GUDHI

This notebook implements the pipeline from Thesis Section 3.4: simulate an AR($p$) process, split it into training and test data, learn the least-squares completion $\widehat A_k$, lift every window to $\mathbb R^{k_{\max}}$, build a GUDHI flag complex, assemble the scale-similarity matrix, and select $k^*$ from its principal eigenvector.

The graph weight is the distance $w_k(s,t)=\lVert p_s^{(k)}-p_t^{(k)}\rVert_1$. This matches the edge rule $w_k(s,t)\leq\tau$. The reciprocal expression in the current thesis draft would instead require reversing the inequality. One global threshold $\tau$ is estimated from the pooled distances so that it is comparable across $k$.

In [23]:
from dataclasses import dataclass
from typing import Dict, Iterable, Tuple

import gudhi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Function definitions

All reusable definitions are collected before the editable controls and execution cells.

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    p: int = 2
    k_max: int = 5
    n_observations: int = 500
    train_fraction: float = 0.60
    burn_in: int = 500
    innovation_sd: float = 1.0
    seed: int = 20260707
    pacf_first: float = 0.75
    pacf_decay: float = -0.50
    tau: float = 2.0
    max_simplex_dimension: int = 2
    alpha: float = 1.0
    xi: float = 1.0
    tilde_delta: float = 0.0
    similarity_mode: str = "union"  # 'union' or 'overlap'

    def __post_init__(self):
        if self.p < 1:
            raise ValueError("p must be a positive integer.")
        if self.k_max < self.p:
            raise ValueError("k_max must be at least p.")
        if self.tau <= 0:
            raise ValueError("tau must be strictly positive.")

### 1.1 AR simulation and automatic coefficients

In [ ]:
def pacf_to_ar(partial_autocorrelations: Iterable[float]) -> np.ndarray:
    """Convert partial autocorrelations to stationary AR coefficients."""
    phi = np.empty(0, dtype=float)
    for kappa in np.asarray(tuple(partial_autocorrelations), dtype=float):
        if abs(kappa) >= 1:
            raise ValueError("Every partial autocorrelation must lie in (-1, 1).")
        previous = phi.copy()
        phi = np.empty(previous.size + 1)
        if previous.size:
            phi[:-1] = previous - kappa * previous[::-1]
        phi[-1] = kappa
    return phi


def automatic_ar_coefficients(order: int, first: float = 0.75, decay: float = -0.50) -> np.ndarray:
    pacf = first * decay ** np.arange(order)
    return pacf_to_ar(pacf)


def companion_radius(phi: np.ndarray) -> float:
    companion = np.zeros((len(phi), len(phi)))
    companion[0] = phi
    if len(phi) > 1:
        companion[1:, :-1] = np.eye(len(phi) - 1)
    return float(np.max(np.abs(np.linalg.eigvals(companion))))


def simulate_ar(phi: np.ndarray, n: int, burn_in: int, innovation_sd: float, rng: np.random.Generator) -> np.ndarray:
    phi = np.asarray(phi, dtype=float)
    radius = companion_radius(phi)
    if radius >= 1:
        raise ValueError(f"AR coefficients are not stationary (spectral radius={radius:.4f}).")
    p = len(phi)
    eps = rng.normal(0.0, innovation_sd, n + burn_in + p)
    x = np.zeros_like(eps)
    for t in range(p, x.size):
        x[t] = phi @ x[t - np.arange(1, p + 1)] + eps[t]
    return x[burn_in + p:]

### 1.2 Least-squares dimension lift

For each $k$, training windows estimate $\widehat A_k$ in $z_t^{(k)}\approx A_kY_t^{(k)}$. Test windows are aligned across all scales and lifted to the common dimension $k_{\max}$.

In [ ]:
def aligned_windows(x: np.ndarray, width: int) -> np.ndarray:
    if len(x) < width:
        raise ValueError(f"Need at least {width} observations, received {len(x)}.")
    return np.lib.stride_tricks.sliding_window_view(np.asarray(x), width)


def fit_completion_matrix(train: np.ndarray, k: int, k_max: int) -> np.ndarray:
    full = aligned_windows(train, k_max)
    if k == k_max:
        return np.empty((0, k))
    y, z = full[:, :k], full[:, k:]
    coefficients, *_ = np.linalg.lstsq(y, z, rcond=None)
    return coefficients.T


def lift_test_cloud(test: np.ndarray, k: int, k_max: int, completion: np.ndarray) -> np.ndarray:
    y = aligned_windows(test, k_max)[:, :k]
    predicted = y @ completion.T
    return np.concatenate((y, predicted), axis=1)


def build_lifted_clouds(train: np.ndarray, test: np.ndarray, k_max: int):
    completions, clouds = {}, {}
    for k in range(1, k_max + 1):
        completions[k] = fit_completion_matrix(train, k, k_max)
        clouds[k] = lift_test_cloud(test, k, k_max, completions[k])
        assert clouds[k].shape[1] == k_max
    return completions, clouds

### 1.3 $L^1$ graph weights and GUDHI flag complexes

The manually selected value $\tau$ is shared across all scales. GUDHI stores vertices and all edges with $L^1$ distance at most $\tau$, then expands the graph to its flag complex.

In [ ]:
def l1_distance_matrices(clouds: Dict[int, np.ndarray]) -> Dict[int, np.ndarray]:
    return {k: squareform(pdist(points, metric="cityblock")) for k, points in clouds.items()}


def gudhi_flag_complex(distance_matrix: np.ndarray, tau: float, max_dimension: int = 2) -> gudhi.SimplexTree:
    n = distance_matrix.shape[0]
    complex_ = gudhi.SimplexTree()
    for vertex in range(n):
        complex_.insert([vertex], filtration=0.0)
    rows, cols = np.where(np.triu(distance_matrix <= tau, k=1))
    for i, j in zip(rows, cols):
        complex_.insert([int(i), int(j)], filtration=float(distance_matrix[i, j]))
    complex_.expansion(max_dimension)
    complex_.make_filtration_non_decreasing()
    return complex_


def union_complex(left: gudhi.SimplexTree, right: gudhi.SimplexTree) -> gudhi.SimplexTree:
    union = gudhi.SimplexTree()
    for tree in (left, right):
        for simplex, filtration in tree.get_filtration():
            union.insert(simplex, filtration=filtration)
    union.make_filtration_non_decreasing()
    return union


def complex_counts(complex_: gudhi.SimplexTree) -> Tuple[int, int]:
    # beta_0 is computed by GUDHI persistence; |E| comes from its 1-skeleton.
    complex_.compute_persistence(persistence_dim_max=True)
    betti = complex_.betti_numbers()
    beta_0 = int(betti[0]) if betti else complex_.num_vertices()
    edges = sum(1 for simplex, _ in complex_.get_skeleton(1) if len(simplex) == 2)
    return beta_0, edges


def log_functional(complex_: gudhi.SimplexTree, alpha: float, xi: float, tilde_delta: float) -> float:
    beta_0, edges = complex_counts(complex_)
    # The max(1, |E|) convention keeps the exploratory log-functional defined for an edgeless graph.
    return float(tilde_delta + alpha * np.log(beta_0) + xi * np.log(max(1, edges)))

### 1.4 Similarity matrix and spectral selection

The default thesis candidate is $\Sigma_{ij}=\widetilde f(L_i\cup L_j)$, which is nonnegative here. The alternative `overlap` mode uses $\widetilde f(L_i)+\widetilde f(L_j)-\widetilde f(L_i\cup L_j)$; it is retained for experimentation but is not guaranteed to be nonnegative because the exploratory log-functional has not been proved subadditive.

In [ ]:
def scale_similarity_matrix(complexes: Dict[int, gudhi.SimplexTree], config: PipelineConfig):
    ks = sorted(complexes)
    individual = {k: log_functional(complexes[k], config.alpha, config.xi, config.tilde_delta) for k in ks}
    sigma = np.empty((len(ks), len(ks)))
    for a, i in enumerate(ks):
        for b in range(a, len(ks)):
            j = ks[b]
            f_union = log_functional(union_complex(complexes[i], complexes[j]), config.alpha, config.xi, config.tilde_delta)
            value = f_union if config.similarity_mode == "union" else individual[i] + individual[j] - f_union
            sigma[a, b] = sigma[b, a] = value
    if config.similarity_mode not in {"union", "overlap"}:
        raise ValueError("similarity_mode must be 'union' or 'overlap'.")
    return sigma, individual


def principal_scale(sigma: np.ndarray, ks: Iterable[int]):
    eigenvalues, eigenvectors = np.linalg.eigh((sigma + sigma.T) / 2)
    vector = eigenvectors[:, -1]
    if vector.sum() < 0:
        vector = -vector
    if np.min(vector) > -1e-10:
        vector = np.maximum(vector, 0)
    vector = vector / np.sum(np.abs(vector))
    ks = np.asarray(tuple(ks))
    return int(ks[np.argmax(vector)]), eigenvalues, vector


def run_pipeline(order: int, config: PipelineConfig, seed_offset: int = 0):
    phi = automatic_ar_coefficients(order, config.pacf_first, config.pacf_decay)
    rng = np.random.default_rng(config.seed + seed_offset)
    x = simulate_ar(phi, config.n_observations, config.burn_in, config.innovation_sd, rng)
    split = int(config.train_fraction * len(x))
    train, test = x[:split], x[split:]
    completions, clouds = build_lifted_clouds(train, test, config.k_max)
    distances = l1_distance_matrices(clouds)
    tau = float(config.tau)
    complexes = {k: gudhi_flag_complex(d, tau, config.max_simplex_dimension) for k, d in distances.items()}
    sigma, functionals = scale_similarity_matrix(complexes, config)
    k_star, eigenvalues, eigenvector = principal_scale(sigma, sorted(complexes))
    counts = {k: complex_counts(complexes[k]) for k in sorted(complexes)}
    return {
        "order": order, "phi": phi, "series": x, "train": train, "test": test,
        "completions": completions, "clouds": clouds, "distances": distances,
        "tau": tau, "complexes": complexes, "counts": counts,
        "functionals": functionals, "sigma": sigma, "eigenvalues": eigenvalues,
        "eigenvector": eigenvector, "k_star": k_star,
    }

### 1.5 Experiment helper: single-process $\tau$ scan

In [ ]:
def scan_tau_single_process(
    order: int,
    config: PipelineConfig,
    tau_values: Iterable[float],
    seed: int = 20260801,
):
    """Scan tau for one simulated AR process and retain every curve."""
    tau_values = np.asarray(tuple(tau_values), dtype=float)
    if tau_values.ndim != 1 or tau_values.size == 0 or np.any(tau_values <= 0):
        raise ValueError("tau_values must be a nonempty sequence of positive numbers.")

    phi = automatic_ar_coefficients(order, config.pacf_first, config.pacf_decay)
    rng = np.random.default_rng(seed)
    x = simulate_ar(phi, config.n_observations, config.burn_in, config.innovation_sd, rng)
    split = int(config.train_fraction * len(x))
    completions, clouds = build_lifted_clouds(x[:split], x[split:], config.k_max)
    distances = l1_distance_matrices(clouds)
    ks = np.arange(1, config.k_max + 1)
    n_vertices = next(iter(distances.values())).shape[0]
    possible_edges = n_vertices * (n_vertices - 1) / 2

    records = []
    curves = {}
    for tau_value in tau_values:
        complexes = {
            k: gudhi_flag_complex(distance, float(tau_value), max_dimension=1)
            for k, distance in distances.items()
        }
        sigma, functionals = scale_similarity_matrix(complexes, config)
        k_star, eigenvalues, eigenvector = principal_scale(sigma, ks)
        leading = float(eigenvalues[-1])
        relative_gap = (
            float((eigenvalues[-1] - eigenvalues[-2]) / abs(leading))
            if len(eigenvalues) > 1 and leading != 0
            else np.nan
        )
        counts = {k: complex_counts(complexes[k]) for k in ks}
        beta_0_curve = np.array([counts[k][0] for k in ks], dtype=float)
        edge_curve = np.array([counts[k][1] for k in ks], dtype=float)
        beta_contribution = config.alpha * np.log(beta_0_curve)
        edge_contribution = config.xi * np.log(np.maximum(1.0, edge_curve))
        mean_edge_density = float(np.mean(edge_curve / possible_edges))
        functional_curve = np.array([functionals[k] for k in ks])
        if not np.allclose(
            functional_curve,
            config.tilde_delta + beta_contribution + edge_contribution,
        ):
            raise RuntimeError("Functional decomposition is inconsistent.")
        curves[float(tau_value)] = {
            "functional": functional_curve,
            "perron": eigenvector,
            "beta_0": beta_0_curve,
            "edges": edge_curve,
            "beta_contribution": beta_contribution,
            "edge_contribution": edge_contribution,
            "sigma": sigma,
            "k_star": k_star,
        }
        records.append({
            "tau": float(tau_value),
            "k_star": k_star,
            "correct": k_star == order,
            "relative_spectral_gap": relative_gap,
            "mean_edge_density": mean_edge_density,
        })

    return {
        "order": order,
        "phi": phi,
        "series": x,
        "completions": completions,
        "clouds": clouds,
        "distances": distances,
        "results": pd.DataFrame.from_records(records),
        "curves": curves,
    }

## 2. Controls

Edit only the five values below. The manually chosen $\tau$ is used at every scale. The default is $\widetilde\delta=0$, and stable, nonzero AR coefficients are generated automatically from the selected order $p$.

In [ ]:
# ========================= USER CONTROLS =========================
p = 1
k_max = 5
tau = 2.0
alpha = 1.0
xi = 1.0
# ================================================================

CONFIG = PipelineConfig(p=p, k_max=k_max, tau=tau, alpha=alpha, xi=xi)
CONFIG

## 3. Run the selected AR($p$) model

The next cell prints the automatically generated coefficients and displays the scale-similarity matrix as a labeled pandas table.

In [ ]:
result = run_pipeline(CONFIG.p, CONFIG)
ks = np.arange(1, CONFIG.k_max + 1)

print(f"AR({CONFIG.p}) coefficients: {np.round(result['phi'], 5)}")
print(f"Common L1 threshold tau: {result['tau']:.5f}")
print(f"Selected scale k*: {result['k_star']}")

### Functional and principal-eigenvector plots

In [ ]:
functional_values = np.array([result["functionals"][k] for k in ks])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)

axes[0].plot(ks, functional_values, marker="o", linewidth=2, color="tab:blue")
axes[0].axvline(CONFIG.p, color="tab:green", linestyle="--", label="true $p$")
axes[0].set(
    title=rf"Functional $\widetilde f(L_k)$ for AR({CONFIG.p})",
    xlabel="scale $k$", ylabel=rf"$\widetilde f= {CONFIG.alpha:g}\log\beta_0 + {CONFIG.xi:g}\log|E|$", xticks=ks,
)
axes[0].legend()

axes[1].plot(ks, result["eigenvector"], marker="o", linewidth=2, color="tab:purple")
axes[1].axvline(CONFIG.p, color="tab:green", linestyle="--", label="true $p$")
axes[1].axvline(result["k_star"], color="tab:red", linestyle=":", label="selected $k^*$")
axes[1].set(
    title=r"Principal eigenvector of $\Sigma$",
    xlabel="scale $k$", ylabel="normalized eigenvector entry", xticks=ks,
)
axes[1].legend()

plt.show()

### Scale-similarity matrix

In [ ]:
similarity_df = pd.DataFrame(
    result["sigma"],
    index=pd.Index(ks, name="k_i"),
    columns=pd.Index(ks, name="k_j"),
)
display(
    similarity_df.style
    .format(precision=4)
    .background_gradient(cmap="Blues")
    .set_caption(r"Scale-similarity matrix $\Sigma_{k_{\max}}$")
)

### Histograms of the $L^1$ weights

Each vertical panel uses only pairs $s<t$, so every unordered pair appears exactly once. Common bin edges make the distributions comparable across scales; the dashed line is the manually chosen threshold $\tau$.

In [ ]:
upper_triangle_weights = {
    k: distance[np.triu_indices_from(distance, k=1)]
    for k, distance in result["distances"].items()
}
pooled_weights = np.concatenate(list(upper_triangle_weights.values()))
common_bins = np.histogram_bin_edges(pooled_weights, bins=35)

fig, axes = plt.subplots(
    CONFIG.k_max,
    1,
    figsize=(10, 2.5 * CONFIG.k_max),
    sharex=True,
    constrained_layout=True,
)
axes = np.atleast_1d(axes)

for axis, k in zip(axes, ks):
    weights = upper_triangle_weights[k]
    axis.hist(weights, bins=common_bins, color="tab:blue", alpha=0.78, edgecolor="white")
    axis.axvline(CONFIG.tau, color="tab:red", linestyle="--", linewidth=1.8, label=rf"$\tau={CONFIG.tau:g}$")
    axis.set_title(rf"$k={k}$: $w_k(s,t)$ for $s<t$")
    axis.set_ylabel("count")
    axis.legend(loc="upper right")

axes[-1].set_xlabel(r"$w_k(s,t)=\|p_s^{(k)}-p_t^{(k)}\|_1$")
fig.suptitle(r"Distributions of the $L^1$ graph weights", fontsize=14)
plt.show()

## 4. Re-running with another order

Change the five values in the control cell and run from that cell downward. The current result is one realization, not evidence of order-recovery performance; a later Monte Carlo layer should measure how often $k^*=p$ over independent seeds.

## 5. Experiment 1: scan $\tau$ for one AR(1) process

A single AR(1) path is simulated and held fixed throughout the experiment. We vary
\[
\tau\in\{0.5,1.0,\ldots,10.0\},
\]
while inheriting $k_{\max}$, $\alpha$, and $\xi$ from the main control cell. For each threshold, four curves are shown in one row: the functional, the Perron vector, the raw $\beta_0$ and edge counts, and their additive contributions $\alpha\log\beta_0$ and $\xi\log|E|$. Thus changes across rows are caused only by $\tau$, not by a newly simulated process.

Only the GUDHI 1-skeleton is required here because the current functional uses $\beta_0$ and the number of edges.

In [ ]:
experiment_tau_values = np.arange(0.5, 10.0 + 0.25, 0.5)
experiment_seed = 20260801

experiment = scan_tau_single_process(
    order=1,
    config=CONFIG,
    tau_values=experiment_tau_values,
    seed=experiment_seed,
)
experiment_results = experiment["results"]
experiment_curves = experiment["curves"]

print(
    f"Experiment 1: one AR(1) process, k_max={CONFIG.k_max}, "
    f"alpha={CONFIG.alpha:g}, xi={CONFIG.xi:g}"
)
display(
    experiment_results.style
    .format({
        "tau": "{:.1f}",
        "relative_spectral_gap": "{:.4f}",
        "mean_edge_density": "{:.3f}",
    })
    .map(lambda value: "background-color: #d9ead3" if value else "", subset=["correct"])
    .set_caption(r"Single-process $\tau$ scan for AR(1)")
)

In [ ]:
experiment_ks = np.arange(1, CONFIG.k_max + 1)
number_of_tau_values = len(experiment_tau_values)

fig, axes = plt.subplots(
    number_of_tau_values,
    4,
    figsize=(24, 2.8 * number_of_tau_values),
    sharex=True,
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

for row, tau_value in enumerate(experiment_tau_values):
    curve = experiment_curves[float(tau_value)]
    functional_axis, perron_axis, counts_axis, contributions_axis = axes[row]

    functional_axis.plot(
        experiment_ks,
        curve["functional"],
        marker="o",
        linewidth=2,
        color="tab:blue",
    )
    functional_axis.axvline(1, color="tab:green", linestyle="--", label="true $p=1$")
    functional_axis.set_ylabel(rf"$\tau={tau_value:.1f}$")
    functional_axis.grid(alpha=0.3)

    perron_axis.plot(
        experiment_ks,
        curve["perron"],
        marker="o",
        linewidth=2,
        color="tab:purple",
    )
    perron_axis.axvline(1, color="tab:green", linestyle="--", label="true $p=1$")
    perron_axis.axvline(
        curve["k_star"],
        color="tab:red",
        linestyle=":",
        label=rf"selected $k^*={curve['k_star']}$",
    )
    perron_axis.grid(alpha=0.3)
    perron_axis.legend(loc="best", fontsize=8)

    beta_line = counts_axis.plot(
        experiment_ks,
        curve["beta_0"],
        marker="o",
        linewidth=2,
        color="tab:blue",
        label=r"$\beta_0$",
    )
    edges_axis = counts_axis.twinx()
    edge_line = edges_axis.plot(
        experiment_ks,
        curve["edges"],
        marker="s",
        linewidth=2,
        color="tab:orange",
        label=r"$|E|$",
    )
    counts_axis.set_ylabel(r"$\beta_0$", color="tab:blue")
    edges_axis.set_ylabel(r"$|E|$", color="tab:orange")
    counts_axis.tick_params(axis="y", colors="tab:blue")
    edges_axis.tick_params(axis="y", colors="tab:orange")
    counts_axis.grid(alpha=0.3)
    counts_axis.legend(beta_line + edge_line, [line.get_label() for line in beta_line + edge_line], loc="best", fontsize=8)

    beta_part_line = contributions_axis.plot(
        experiment_ks,
        curve["beta_contribution"],
        marker="o",
        linewidth=2,
        color="tab:green",
        label=r"$\alpha\log\beta_0$",
    )
    edge_part_axis = contributions_axis.twinx()
    edge_part_line = edge_part_axis.plot(
        experiment_ks,
        curve["edge_contribution"],
        marker="s",
        linewidth=2,
        color="tab:red",
        label=r"$\xi\log|E|$",
    )
    contributions_axis.set_ylabel(r"$\alpha\log\beta_0$", color="tab:green")
    edge_part_axis.set_ylabel(r"$\xi\log|E|$", color="tab:red")
    contributions_axis.tick_params(axis="y", colors="tab:green")
    edge_part_axis.tick_params(axis="y", colors="tab:red")
    contributions_axis.grid(alpha=0.3)
    contributions_axis.legend(
        beta_part_line + edge_part_line,
        [line.get_label() for line in beta_part_line + edge_part_line],
        loc="best",
        fontsize=8,
    )

axes[0, 0].set_title(r"Functional $\widetilde f(L_k)$")
axes[0, 1].set_title(r"Perron vector $v_k^*$")
axes[0, 2].set_title(r"Raw counts: $\beta_0$ and $|E|$")
axes[0, 3].set_title(r"Contributions: $\alpha\log\beta_0$ and $\xi\log|E|$")
for column in range(4):
    axes[-1, column].set_xlabel("scale $k$")
for axis in axes.flat:
    axis.set_xticks(experiment_ks)

fig.suptitle(
    rf"Single AR(1) process: $\tau=0.5,1.0,\ldots,10.0$ "
    rf"($k_{{\max}}={CONFIG.k_max}$, $\alpha={CONFIG.alpha:g}$, $\xi={CONFIG.xi:g}$)",
    fontsize=15,
)
plt.show()